In [ ]:
"""
BPE Tokenizer — CUDA GPU Edition  (memory-efficient, NVRTC-free)
=================================================================
Compatible with Google Colab / Tesla T4 / any CuPy install.
No RawModule / RawKernel — zero NVRTC dependency.

OOM fix vs previous version
----------------------------
The broadcast match matrix  (n-1) % W  has been replaced by a
sorted-table lookup using cp.searchsorted, which is O(n log W) time
and O(n + W) memory — safe even for n=200 000, W=500.
"""

from __future__ import annotations

import time
import random
import collections
import heapq
from typing import Dict, List, Tuple

try:
    import cupy as cp
    import numpy as np
    CUDA_AVAILABLE = True
except ImportError:
    import numpy as np
    CUDA_AVAILABLE = False
    print("[WARN] CuPy not found. Install cupy-cuda12x.")

def generate_long_string(length: int = 200_000) -> str:
    random.seed(42)
    alphabet = "aabbccddeeffgghhiijjkkllmmnnooppqqrrssttuuvvwwxxyyzz "
    return "".join(random.choice(alphabet) for _ in range(length))

def gpu_count_pairs(d_tokens: "cp.ndarray", vocab_cap: int) -> Dict[Tuple[int, int], int]:
    n = len(d_tokens)
    if n < 2:
        return {}
    lefts  = d_tokens[:-1].astype(cp.int64)
    rights = d_tokens[1: ].astype(cp.int64)
    keys   = lefts * vocab_cap + rights
    unique_keys, counts = cp.unique(keys, return_counts=True)
    uk = unique_keys.get()
    uc = counts.get()
    result: Dict[Tuple[int, int], int] = {}
    for k, c in zip(uk, uc):
        result[(int(k) // vocab_cap, int(k) % vocab_cap)] = int(c)
    return result

def gpu_apply_wave(d_tokens: "cp.ndarray", wave: List[Tuple[Tuple[int, int], int]], vocab_cap: int) -> "cp.ndarray":
    n = len(d_tokens)
    if n < 2 or not wave:
        return d_tokens
    lefts     = d_tokens[:-1].astype(cp.int64)
    rights    = d_tokens[1: ].astype(cp.int64)
    pair_keys = lefts * vocab_cap + rights
    wave_keys_np = np.array([p[0] * vocab_cap + p[1] for (p, _) in wave], dtype=np.int64)
    wave_vals_np = np.array([nid for (_, nid) in wave], dtype=np.int32)
    sort_order   = np.argsort(wave_keys_np)
    wave_keys_np = wave_keys_np[sort_order]
    wave_vals_np = wave_vals_np[sort_order]
    d_wave_keys = cp.asarray(wave_keys_np)
    d_wave_vals = cp.asarray(wave_vals_np)
    idx = cp.searchsorted(d_wave_keys, pair_keys, side='left')
    in_range  = idx < len(d_wave_keys)
    idx_safe  = cp.clip(idx, 0, len(d_wave_keys) - 1)
    exact     = d_wave_keys[idx_safe] == pair_keys
    hit_mask  = in_range & exact
    out_tokens = d_tokens.copy().astype(cp.int32)
    alive      = cp.ones(n, dtype=cp.bool_)
    hit_positions   = cp.where(hit_mask)[0]
    if hit_positions.size > 0:
        new_ids_at_hits = d_wave_vals[idx_safe[hit_positions]]
        out_tokens[hit_positions] = new_ids_at_hits
        right_positions = hit_positions + 1
        right_positions = right_positions[right_positions < n]
        alive[right_positions] = False
    alive_int = alive.view(cp.int8)
    scan      = cp.cumsum(alive_int).astype(cp.int32) - 1
    new_len   = int(alive_int.sum())
    out_compact = cp.empty(new_len, dtype=cp.int32)
    out_compact[scan[alive]] = out_tokens[alive]
    return out_compact

def _speculative_select_gpu(tokens_orig: List[int], num_merges: int, base_next_id: int, vocab_cap: int) -> Tuple[list, dict]:
    d_tok          = cp.asarray(np.array(tokens_orig, dtype=np.int32))
    pair_freq_dict = gpu_count_pairs(d_tok, vocab_cap)
    shadow = list(tokens_orig)
    dead = set()
    pair_locs = collections.defaultdict(list)
    for i in range(len(tokens_orig) - 1):
        pair_locs[(tokens_orig[i], tokens_orig[i+1])].append(i)
    heap = []
    counter = 0
    for pair, freq in pair_freq_dict.items():
        heapq.heappush(heap, (-freq, counter, pair))
        counter += 1
    invalidated = set()
    next_id = base_next_id
    merge_order, new_vocab = [], {}
    for _ in range(num_merges):
        best_pair = None
        while heap:
            _, _, p = heapq.heappop(heap)
            if p in invalidated: continue
            act = sum(1 for pos in pair_locs[p] if pos not in dead and pos+1 not in dead and shadow[pos]==p[0] and shadow[pos+1]==p[1])
            if act >= 2:
                best_pair = p; break
        if not best_pair: break
        nid = next_id; next_id += 1
        new_vocab[nid] = best_pair
        merge_order.append((best_pair, nid))
        invalidated.add(best_pair)
        for pos in [pos for pos in pair_locs[best_pair] if pos not in dead and pos+1 not in dead and shadow[pos]==best_pair[0] and shadow[pos+1]==best_pair[1]]:
            lp, rp = pos - 1, pos + 2
            while lp >= 0 and lp in dead: lp -= 1
            while rp < len(shadow) and rp in dead: rp += 1
            if lp >= 0: invalidated.add((shadow[lp], shadow[pos]))
            if rp < len(shadow): invalidated.add((shadow[pos+1], shadow[rp]))
            shadow[pos] = nid; dead.add(pos+1)
            if lp >= 0: pair_locs[(shadow[lp], nid)].append(lp)
            if rp < len(shadow): pair_locs[(nid, shadow[rp])].append(pos)
    return merge_order, new_vocab

def _build_dag(merge_order: list) -> dict:
    prod, deps = {}, {i: set() for i in range(len(merge_order))}
    for idx, (pair, nt) in enumerate(merge_order):
        for t in pair:
            if t in prod: deps[idx].add(prod[t])
        prod[nt] = idx
    return deps

def _topo_levels(deps: dict) -> list:
    n = len(deps)
    level, children, in_deg = [-1]*n, collections.defaultdict(list), [0]*n
    for node, parents in deps.items():
        for p in parents: children[p].append(node); in_deg[node] += 1
    q = collections.deque(node for node in range(n) if in_deg[node] == 0)
    for node in q: level[node] = 0
    while q:
        u = q.popleft()
        for v in children[u]:
            in_deg[v] -= 1
            level[v] = max(level[v], level[u] + 1)
            if in_deg[v] == 0: q.append(v)
    if not level: return []
    max_lv = max(level)
    levels = [[] for _ in range(max_lv + 1)]
    for node, lv in enumerate(level): levels[lv].append(node)
    return levels

def standard_bpe(text, num_merges):
    c2i, i2s, tokens = {}, {}, []
    for c in text:
        if c not in c2i: tid = len(c2i); c2i[c] = tid; i2s[tid] = c
        tokens.append(c2i[c])
    next_id, merges, vocab = len(c2i), [], dict(i2s)
    for _ in range(num_merges):
        f = collections.defaultdict(int)
        for i in range(len(tokens)-1): f[(tokens[i], tokens[i+1])] += 1
        if not f: break
        best = max(f, key=f.get)
        if f[best] < 2: break
        nid = next_id; next_id += 1
        vocab[nid] = vocab[best[0]] + vocab[best[1]]
        merges.append((best, nid))
        new_t, i = [], 0
        while i < len(tokens):
            if i<(len(tokens)-1) and tokens[i]==best[0] and tokens[i+1]==best[1]:
                new_t.append(nid); i += 2
            else: new_t.append(tokens[i]); i += 1
        tokens = new_t
    return vocab, merges, tokens

def cuda_bpe(text, num_merges):
    c2i, i2s, tokens = {}, {}, []
    for c in text:
        if c not in c2i: tid = len(c2i); c2i[c] = tid; i2s[tid] = c
        tokens.append(c2i[c])
    base_id, vocab = len(c2i), dict(i2s)
    v_cap = base_id + num_merges + 10
    m_order, n_vocab = _speculative_select_gpu(tokens, num_merges, base_id, v_cap)
    def res(t):
        if t in vocab: return vocab[t]
        l, r = n_vocab[t]
        return res(l) + res(r)
    for nid in sorted(n_vocab): vocab[nid] = res(nid)
    deps, levels = _build_dag(m_order), _topo_levels(_build_dag(m_order))
    d_tokens = cp.asarray(np.array(tokens, dtype=np.int32))
    for lv in levels:
        wave = [m_order[i] for i in lv]
        d_tokens = gpu_apply_wave(d_tokens, wave, v_cap)
    return vocab, m_order, d_tokens.get().tolist(), len(levels)

def benchmark(text, num_merges):
    print("="*65 + "\n  BPE BENCHMARK\n" + "="*65)
    t0 = time.perf_counter(); sv, sm, st = standard_bpe(text, num_merges); t_std = time.perf_counter()-t0
    print(f"Standard: {t_std:.4f}s")
    _ = cuda_bpe(text[:500], 5)
    t0 = time.perf_counter(); gv, gm, gt, dd = cuda_bpe(text, num_merges); t_gpu = time.perf_counter()-t0
    print(f"CUDA: {t_gpu:.4f}s\nSpeedup: {t_std/t_gpu:.2f}x\n" + "="*65)
    return {"std_time": t_std, "gpu_time": t_gpu}

if __name__ == "__main__":
    TEXT = generate_long_string(200_000)
    benchmark(TEXT, 500)

  BPE BENCHMARK
Standard: 34.7254s
CUDA: 0.4970s
Speedup: 69.87x


In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset
import time

print("Loading OpenWebText sample (streaming to avoid script issues)...")
# Using the main 'openwebtext' dataset in streaming mode to quickly grab a sample
ds = load_dataset("openwebtext", split="train", streaming=True)

# Grab the first 50 examples
text_list = []
it = iter(ds)
for _ in range(600):
    try:
        text_list.append(next(it)['text'])
    except StopIteration:
        break

owt_text = " ".join(text_list)
print(f"Test text length: {len(owt_text):,} characters")

def run_parallel_only_benchmark(text, num_merges):
    print("="*65)
    print("  PARALLEL CUDA BPE BENCHMARK (OpenWebText)")
    print("="*65)

    # Warm up
    _ = cuda_bpe(text[:500], 5)

    t0 = time.perf_counter()
    vocab, merges, tokens, depth = cuda_bpe(text, num_merges)
    t_gpu = time.perf_counter() - t0

    print(f"CUDA Time: {t_gpu:.4f}s")
    print(f"Tokens produced: {len(tokens):,}")
    print(f"DAG Depth: {depth}")
    print("="*65)

run_parallel_only_benchmark(owt_text, 1000)

Loading OpenWebText sample (streaming to avoid script issues)...


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Test text length: 2,999,986 characters
  PARALLEL CUDA BPE BENCHMARK (OpenWebText)
CUDA Time: 3.3763s
Tokens produced: 2,041,827
DAG Depth: 1
